# Data Cleaning & Preprocessing

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## 1. Set Project Paths

In [2]:
PROJECT_ROOT = Path(__file__).resolve().parent.parent
RAW_PATH = PROJECT_ROOT / "data" / "raw"
CLEANED_PATH = PROJECT_ROOT / "data" / "cleaned"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_PATH)
print("Cleaned data path:", CLEANED_PATH)


Project root: d:\HCL-GUVI\Cart_to_Insights
Raw data path: d:\HCL-GUVI\Cart_to_Insights\data\raw
Cleaned data path: d:\HCL-GUVI\Cart_to_Insights\data\cleaned


## 2. Load Raw Data

In [3]:
file_names = {
    "Customers": "olist_customers_dataset.csv",
    "Geolocation": "olist_geolocation_dataset.csv",
    "Orders": "olist_orders_dataset.csv",
    "Order Items": "olist_order_items_dataset.csv",
    "Order Payments": "olist_order_payments_dataset.csv",
    "Order Reviews": "olist_order_reviews_dataset.csv",
    "Products": "olist_products_dataset.csv",
    "Sellers": "olist_sellers_dataset.csv",
    "Category Translation": "product_category_name_translation.csv"
}

raw_tables = {}

for table_name, file_name in file_names.items():
    file_path = RAW_PATH / file_name
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    raw_tables[table_name] = pd.read_csv(file_path)

customers = raw_tables["Customers"]
geolocation = raw_tables["Geolocation"]
orders = raw_tables["Orders"]
order_items = raw_tables["Order Items"]
order_payments = raw_tables["Order Payments"]
order_reviews = raw_tables["Order Reviews"]
products = raw_tables["Products"]
sellers = raw_tables["Sellers"]
category_translation = raw_tables["Category Translation"]

tables = raw_tables.copy()

print("All raw datasets loaded successfully.")


All raw datasets loaded successfully.


## 3. Dataset Summary

In [4]:
summary = []

for table_name, table in tables.items():
    summary.append({
        "Table": table_name,
        "Rows": table.shape[0],
        "Columns": table.shape[1]
    })

summary_df = pd.DataFrame(summary)
display(summary_df)


,Table,Rows,Columns
0,Customers,99441,5
1,Geolocation,1000163,5
2,Orders,99441,8
3,Order Items,112650,7
4,Order Payments,103886,5
5,Order Reviews,99224,7
6,Products,32951,9
7,Sellers,3095,4
8,Category Translation,71,2


## 4. Dataset Structure

In [5]:
for table_name, table in tables.items():
    print("\n" + "=" * 70)
    print(f"TABLE: {table_name.upper()}")
    print("=" * 70)
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())
    print("\nMissing Values:")
    print(table.isna().sum())
    print("\nData Types:")
    print(table.dtypes)



TABLE: CUSTOMERS
Shape: (99441, 5)
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Missing Values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Data Types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

TABLE: GEOLOCATION
Shape: (1000163, 5)
Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Missing Values:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Data Types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                

## 5. Missing Value Analysis

In [6]:
missing_report = []

for table_name, table in tables.items():
    for column in table.columns:
        missing_count = table[column].isna().sum()
        missing_percentage = (missing_count / len(table)) * 100

        if missing_count > 0:
            missing_report.append({
                "Table Name": table_name,
                "Column Name": column,
                "Missing Count": missing_count,
                "Missing Percentage": round(missing_percentage, 6)
            })

missing_report_df = pd.DataFrame(missing_report)

if not missing_report_df.empty:
    missing_report_df = missing_report_df.sort_values(
        by="Missing Percentage",
        ascending=False
    ).reset_index(drop=True)

display(missing_report_df)


,Table Name,Column Name,Missing Count,Missing Percentage
0,Order Reviews,review_comment_title,87656,88.341530
1,Order Reviews,review_comment_message,58247,58.702532
2,Orders,order_delivered_customer_date,2965,2.981668
3,Products,product_name_lenght,610,1.851234
4,Products,product_category_name,610,1.851234
5,Products,product_description_lenght,610,1.851234
6,Products,product_photos_qty,610,1.851234
7,Orders,order_delivered_carrier_date,1783,1.793023
8,Orders,order_approved_at,160,0.160899
9,Products,product_weight_g,2,0.006070


## 6. Duplicate Analysis

In [7]:
duplicate_report = []

for table_name, table in tables.items():
    duplicate_count = table.duplicated().sum()
    duplicate_percentage = (duplicate_count / len(table)) * 100

    if duplicate_count > 0:
        duplicate_report.append({
            "Table Name": table_name,
            "Duplicate Count": int(duplicate_count),
            "Duplicate Percentage": round(duplicate_percentage, 2)
        })

duplicate_report_df = pd.DataFrame(duplicate_report)

display(duplicate_report_df)


,Table Name,Duplicate Count,Duplicate Percentage
0,Geolocation,261831,26.18


## 7. Data Type Analysis

In [8]:
for table_name, table in tables.items():
    print("\n" + "=" * 70)
    print(f"TABLE: {table_name.upper()}")
    print("=" * 70)
    print(table.dtypes)



TABLE: CUSTOMERS
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

TABLE: GEOLOCATION
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

TABLE: ORDERS
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

TABLE: ORDER ITEMS
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: 

## 8. Date Column Analysis

In [9]:
date_columns = {
    "Orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "Order Reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ],
    "Order Items": [
        "shipping_limit_date"
    ]
}

for table_name, columns in date_columns.items():
    print(f"\n{table_name.upper()}")

    for column in columns:
        print(f"{column} -> {tables[table_name][column].dtype}")



ORDERS
order_purchase_timestamp -> str
order_approved_at -> str
order_delivered_carrier_date -> str
order_delivered_customer_date -> str
order_estimated_delivery_date -> str

ORDER REVIEWS
review_creation_date -> str
review_answer_timestamp -> str

ORDER ITEMS
shipping_limit_date -> str


## 9. Categorical Value Checks

In [10]:
print("ORDER STATUS")
display(orders["order_status"].value_counts())

print("PAYMENT TYPE")
display(order_payments["payment_type"].value_counts())

print("REVIEW SCORE")
display(order_reviews["review_score"].value_counts().sort_index())


ORDER STATUS


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

PAYMENT TYPE


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

REVIEW SCORE


review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

## 10. Review Score Validation

In [11]:
invalid_review_scores = order_reviews[
    ~order_reviews["review_score"].between(1, 5)
]

print("Invalid review scores:", len(invalid_review_scores))


Invalid review scores: 0


## 11. Numerical Value Validation

In [12]:
numeric_checks = {
    "Order Items": [
        "price",
        "freight_value"
    ],
    "Order Payments": [
        "payment_value",
        "payment_installments"
    ],
    "Products": [
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
}

for table_name, columns in numeric_checks.items():
    table = tables[table_name]

    print(f"\n{table_name.upper()}")

    for column in columns:
        negative = (table[column] < 0).sum()
        zero = (table[column] == 0).sum()

        print(
            f"{column}: "
            f"Negative={negative}, "
            f"Zero={zero}"
        )



ORDER ITEMS
price: Negative=0, Zero=0
freight_value: Negative=0, Zero=383

ORDER PAYMENTS
payment_value: Negative=0, Zero=9
payment_installments: Negative=0, Zero=2

PRODUCTS
product_name_lenght: Negative=0, Zero=0
product_description_lenght: Negative=0, Zero=0
product_photos_qty: Negative=0, Zero=0
product_weight_g: Negative=0, Zero=4
product_length_cm: Negative=0, Zero=0
product_height_cm: Negative=0, Zero=0
product_width_cm: Negative=0, Zero=0


## 12. Temporal Consistency Checks

In [13]:
orders_check = orders.copy()

for column in date_columns["Orders"]:
    orders_check[column] = pd.to_datetime(
        orders_check[column],
        errors="coerce"
    )

carrier_before_purchase = (
    orders_check["order_delivered_carrier_date"]
    <
    orders_check["order_purchase_timestamp"]
)

delivery_before_carrier = (
    orders_check["order_delivered_customer_date"]
    <
    orders_check["order_delivered_carrier_date"]
)

delivery_before_purchase = (
    orders_check["order_delivered_customer_date"]
    <
    orders_check["order_purchase_timestamp"]
)

print(
    "Carrier date before purchase:",
    int(carrier_before_purchase.sum())
)

print(
    "Customer delivery before carrier:",
    int(delivery_before_carrier.sum())
)

print(
    "Customer delivery before purchase:",
    int(delivery_before_purchase.sum())
)


Carrier date before purchase: 166
Customer delivery before carrier: 23
Customer delivery before purchase: 0


## 13. Order Date Consistency Checks

In [14]:
delivered_missing_date = orders_check[
    (orders_check["order_status"] == "delivered")
    &
    (orders_check["order_delivered_customer_date"].isna())
]

non_delivered_with_date = orders_check[
    (orders_check["order_status"] != "delivered")
    &
    (orders_check["order_delivered_customer_date"].notna())
]

print(
    "Delivered orders missing customer delivery date:",
    len(delivered_missing_date)
)

print(
    "Non-delivered orders having customer delivery date:",
    len(non_delivered_with_date)
)


Delivered orders missing customer delivery date: 8
Non-delivered orders having customer delivery date: 6


## 14. Primary Key Uniqueness

In [15]:
def check_key_uniqueness(df, column, table_name):
    duplicate_count = df[column].duplicated().sum()
    unique_count = df[column].nunique()

    return {
        "Table": table_name,
        "Column": column,
        "Total Rows": len(df),
        "Unique Values": unique_count,
        "Duplicate Values": int(duplicate_count)
    }

key_results = [
    check_key_uniqueness(customers, "customer_id", "Customers"),
    check_key_uniqueness(orders, "order_id", "Orders"),
    check_key_uniqueness(products, "product_id", "Products"),
    check_key_uniqueness(sellers, "seller_id", "Sellers")
]

key_results_df = pd.DataFrame(key_results)
display(key_results_df)


,Table,Column,Total Rows,Unique Values,Duplicate Values
0,Customers,customer_id,99441,99441,0
1,Orders,order_id,99441,99441,0
2,Products,product_id,32951,32951,0
3,Sellers,seller_id,3095,3095,0


## 15. Composite Key and Identifier Checks

In [16]:
composite_key_results = {
    "Order Items (order_id, order_item_id)": order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum(),

    "Order Payments (order_id, payment_sequential)": order_payments.duplicated(
        subset=["order_id", "payment_sequential"]
    ).sum(),

    "Order Reviews review_id duplicates": order_reviews["review_id"].duplicated().sum(),

    "Customers customer_unique_id duplicates": customers["customer_unique_id"].duplicated().sum()
}

display(
    pd.DataFrame(
        composite_key_results.items(),
        columns=["Check", "Duplicate Count"]
    )
)


,Check,Duplicate Count
0,"Order Items (order_id, order_item_id)",0
1,"Order Payments (order_id, payment_sequential)",0
2,Order Reviews review_id duplicates,814
3,Customers customer_unique_id duplicates,3345


## 16. Outlier Detection

In [17]:
def detect_outliers_iqr(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = df[
        (df[column] < lower_bound)
        |
        (df[column] > upper_bound)
    ]

    return {
        "Column": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outliers": len(outliers)
    }

outlier_results = []

for column in ["price", "freight_value"]:
    outlier_results.append(
        detect_outliers_iqr(order_items, column)
    )

for column in ["payment_value", "payment_installments"]:
    outlier_results.append(
        detect_outliers_iqr(order_payments, column)
    )

outlier_df = pd.DataFrame(outlier_results)
display(outlier_df)


,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outliers
0,price,39.90,134.9000,95.0000,-102.60000,277.40000,8427
1,freight_value,13.08,21.1500,8.0700,0.97500,33.25500,12134
2,payment_value,56.79,171.8375,115.0475,-115.78125,344.40875,7981
3,payment_installments,1.00,4.0000,3.0000,-3.50000,8.50000,6313


## 17. Product Outlier Detection

In [18]:
product_numeric = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

product_outliers = [
    detect_outliers_iqr(products, column)
    for column in product_numeric
]

product_outlier_df = pd.DataFrame(product_outliers)
display(product_outlier_df)


,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outliers
0,product_weight_g,300.0,1900.0,1600.0,-2100.0,4300.0,4551
1,product_length_cm,18.0,38.0,20.0,-12.0,68.0,1380
2,product_height_cm,8.0,21.0,13.0,-11.5,40.5,1892
3,product_width_cm,15.0,30.0,15.0,-7.5,52.5,912


# Step 5: Data Cleaning & Preprocessing

In [19]:
cleaned = {
    name: table.copy()
    for name, table in tables.items()
}


## 18. Rename Product Columns

In [20]:
cleaned["Products"].rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"
    },
    inplace=True
)


## 19. Standardize Identifier Columns

In [21]:
id_columns = {
    "Customers": [
        "customer_id",
        "customer_unique_id"
    ],
    "Orders": [
        "order_id",
        "customer_id"
    ],
    "Order Items": [
        "order_id",
        "product_id",
        "seller_id"
    ],
    "Order Payments": [
        "order_id"
    ],
    "Order Reviews": [
        "review_id",
        "order_id"
    ],
    "Products": [
        "product_id"
    ],
    "Sellers": [
        "seller_id"
    ]
}

for table_name, columns in id_columns.items():
    for column in columns:
        cleaned[table_name][column] = (
            cleaned[table_name][column]
            .astype("string")
            .str.strip()
        )


## 20. Standardize ZIP Code Prefixes

In [22]:
zip_columns = {
    "Customers": "customer_zip_code_prefix",
    "Sellers": "seller_zip_code_prefix",
    "Geolocation": "geolocation_zip_code_prefix"
}

for table_name, column in zip_columns.items():
    cleaned[table_name][column] = (
        pd.to_numeric(
            cleaned[table_name][column],
            errors="coerce"
        )
        .astype("Int64")
        .astype("string")
        .str.zfill(5)
    )


## 21. Convert Date Columns

In [23]:
for table_name, columns in date_columns.items():
    for column in columns:
        cleaned[table_name][column] = pd.to_datetime(
            cleaned[table_name][column],
            errors="coerce"
        )


## 22. Standardize Categorical Columns

In [24]:
categorical_columns = {
    "Customers": [
        "customer_city",
        "customer_state"
    ],
    "Geolocation": [
        "geolocation_city",
        "geolocation_state"
    ],
    "Orders": [
        "order_status"
    ],
    "Order Payments": [
        "payment_type"
    ],
    "Products": [
        "product_category_name"
    ],
    "Sellers": [
        "seller_city",
        "seller_state"
    ],
    "Category Translation": [
        "product_category_name",
        "product_category_name_english"
    ]
}

for table_name, columns in categorical_columns.items():
    for column in columns:
        cleaned[table_name][column] = (
            cleaned[table_name][column]
            .astype("string")
            .str.strip()
            .str.lower()
        )


## 23. Standardize Payment Type

In [25]:
cleaned["Order Payments"]["payment_type"] = (
    cleaned["Order Payments"]["payment_type"]
    .replace({"not_defined": "unknown"})
)


## 24. Handle Review Text Missing Values

In [26]:
cleaned["Order Reviews"]["review_comment_title"] = (
    cleaned["Order Reviews"]["review_comment_title"]
    .fillna("No comment")
)

cleaned["Order Reviews"]["review_comment_message"] = (
    cleaned["Order Reviews"]["review_comment_message"]
    .fillna("No comment")
)


## 25. Handle Product Missing Values

In [27]:
product_numeric_columns = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column in product_numeric_columns:
    median_value = cleaned["Products"][column].median()

    cleaned["Products"][column] = (
        cleaned["Products"][column]
        .fillna(median_value)
    )

cleaned["Products"]["product_category_name"] = (
    cleaned["Products"]["product_category_name"]
    .fillna("unknown")
)


## 26. Remove Exact Duplicate Geolocation Rows

In [28]:
before_geo = len(cleaned["Geolocation"])

cleaned["Geolocation"] = (
    cleaned["Geolocation"]
    .drop_duplicates()
    .reset_index(drop=True)
)

after_geo = len(cleaned["Geolocation"])

print(
    "Geolocation duplicate rows removed:",
    before_geo - after_geo
)


Geolocation duplicate rows removed: 261831


## 27. Handle Invalid Negative Values

In [29]:
non_negative_columns = {
    "Order Items": [
        "price",
        "freight_value"
    ],
    "Order Payments": [
        "payment_value",
        "payment_installments"
    ],
    "Products": [
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
}

for table_name, columns in non_negative_columns.items():
    for column in columns:
        invalid_mask = cleaned[table_name][column] < 0

        cleaned[table_name].loc[
            invalid_mask,
            column
        ] = np.nan


## 28. Validate Review Scores

In [30]:
invalid_review_scores = ~(
    cleaned["Order Reviews"]["review_score"].between(1, 5)
)

cleaned["Order Reviews"].loc[
    invalid_review_scores,
    "review_score"
] = np.nan

print(
    "Invalid review scores handled:",
    int(invalid_review_scores.sum())
)


Invalid review scores handled: 0


## 29. Handle Invalid Order Timestamps

In [31]:
orders_clean = cleaned["Orders"]

carrier_invalid = (
    orders_clean["order_delivered_carrier_date"].notna()
    &
    orders_clean["order_purchase_timestamp"].notna()
    &
    (
        orders_clean["order_delivered_carrier_date"]
        <
        orders_clean["order_purchase_timestamp"]
    )
)

delivery_invalid = (
    orders_clean["order_delivered_customer_date"].notna()
    &
    orders_clean["order_purchase_timestamp"].notna()
    &
    (
        orders_clean["order_delivered_customer_date"]
        <
        orders_clean["order_purchase_timestamp"]
    )
)

delivered_before_carrier = (
    orders_clean["order_delivered_customer_date"].notna()
    &
    orders_clean["order_delivered_carrier_date"].notna()
    &
    (
        orders_clean["order_delivered_customer_date"]
        <
        orders_clean["order_delivered_carrier_date"]
    )
)

non_delivered_with_date = (
    (orders_clean["order_status"] != "delivered")
    &
    orders_clean["order_delivered_customer_date"].notna()
)

carrier_invalid_count = int(carrier_invalid.sum())
delivery_invalid_count = int(delivery_invalid.sum())
delivered_before_carrier_count = int(delivered_before_carrier.sum())
non_delivered_with_date_count = int(non_delivered_with_date.sum())

orders_clean.loc[
    carrier_invalid,
    "order_delivered_carrier_date"
] = pd.NaT

orders_clean.loc[
    delivery_invalid,
    "order_delivered_customer_date"
] = pd.NaT

orders_clean.loc[
    delivered_before_carrier,
    "order_delivered_carrier_date"
] = pd.NaT

orders_clean.loc[
    non_delivered_with_date,
    "order_delivered_customer_date"
] = pd.NaT

print("Carrier dates corrected:", carrier_invalid_count)
print("Customer delivery dates before purchase corrected:", delivery_invalid_count)
print("Customer delivery dates before carrier corrected:", delivered_before_carrier_count)
print("Non-delivered orders with delivery date corrected:", non_delivered_with_date_count)


Carrier dates corrected: 166
Customer delivery dates before purchase corrected: 0
Customer delivery dates before carrier corrected: 23
Non-delivered orders with delivery date corrected: 6


## 30. Cleaned Dataset Summary

In [32]:
cleaning_summary = []

for table_name, table in cleaned.items():
    cleaning_summary.append({
        "Table": table_name,
        "Rows": table.shape[0],
        "Columns": table.shape[1],
        "Missing Values": int(table.isna().sum().sum()),
        "Duplicate Values": int(table.duplicated().sum())
    })

cleaning_summary_df = pd.DataFrame(cleaning_summary)
display(cleaning_summary_df)


,Table,Rows,Columns,Missing Values,Duplicate Values
0,Customers,99441,5,0,0
1,Geolocation,738332,5,0,0
2,Orders,99441,8,5103,0
3,Order Items,112650,7,0,0
4,Order Payments,103886,5,0,0
5,Order Reviews,99224,7,0,0
6,Products,32951,9,0,0
7,Sellers,3095,4,0,0
8,Category Translation,71,2,0,0


## 31. Remaining Missing Values

In [33]:
for table_name, table in cleaned.items():
    missing = table.isna().sum()
    missing = missing[missing > 0]

    print(f"\n{table_name.upper()}")

    if missing.empty:
        print("No missing values")
    else:
        display(missing.to_frame("Missing Count"))



CUSTOMERS
No missing values

GEOLOCATION
No missing values

ORDERS


,Missing Count
order_approved_at,160
order_delivered_carrier_date,1972
order_delivered_customer_date,2971



ORDER ITEMS
No missing values

ORDER PAYMENTS
No missing values

ORDER REVIEWS
No missing values

PRODUCTS
No missing values

SELLERS
No missing values

CATEGORY TRANSLATION
No missing values


## 32. Duplicate Check After Cleaning

In [34]:
duplicate_check = []

for table_name, table in cleaned.items():
    duplicate_check.append({
        "Table": table_name,
        "Duplicate Rows": int(table.duplicated().sum())
    })

duplicate_check_df = pd.DataFrame(duplicate_check)
display(duplicate_check_df)


,Table,Duplicate Rows
0,Customers,0
1,Geolocation,0
2,Orders,0
3,Order Items,0
4,Order Payments,0
5,Order Reviews,0
6,Products,0
7,Sellers,0
8,Category Translation,0


## 33. Data Types After Cleaning

In [35]:
for table_name, table in cleaned.items():
    print(f"\n{table_name.upper()}")
    print(table.dtypes)



CUSTOMERS
customer_id                 string
customer_unique_id          string
customer_zip_code_prefix    string
customer_city               string
customer_state              string
dtype: object

GEOLOCATION
geolocation_zip_code_prefix     string
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                string
geolocation_state               string
dtype: object

ORDERS
order_id                                 string
customer_id                              string
order_status                             string
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

ORDER ITEMS
order_id                       string
order_item_id                   int64
product_id                     string
seller_id                      string
shippin

## 34. Save Cleaned Datasets

In [36]:
CLEANED_PATH.mkdir(parents=True, exist_ok=True)

cleaned_file_names = {
    "Customers": "olist_customers_cleaned.csv",
    "Geolocation": "olist_geolocation_cleaned.csv",
    "Orders": "olist_orders_cleaned.csv",
    "Order Items": "olist_order_items_cleaned.csv",
    "Order Payments": "olist_order_payments_cleaned.csv",
    "Order Reviews": "olist_order_reviews_cleaned.csv",
    "Products": "olist_products_cleaned.csv",
    "Sellers": "olist_sellers_cleaned.csv",
    "Category Translation": "product_category_name_translation_cleaned.csv"
}

for table_name, table in cleaned.items():
    output_file = CLEANED_PATH / cleaned_file_names[table_name]

    table.to_csv(
        output_file,
        index=False
    )

    print(
        f"{table_name}: {len(table):,} rows saved to {output_file}"
    )

print("\nAll cleaned datasets saved successfully.")


Customers: 99,441 rows saved to d:\HCL-GUVI\Cart_to_Insights\data\cleaned\olist_customers_cleaned.csv
Geolocation: 738,332 rows saved to d:\HCL-GUVI\Cart_to_Insights\data\cleaned\olist_geolocation_cleaned.csv
Orders: 99,441 rows saved to d:\HCL-GUVI\Cart_to_Insights\data\cleaned\olist_orders_cleaned.csv
Order Items: 112,650 rows saved to d:\HCL-GUVI\Cart_to_Insights\data\cleaned\olist_order_items_cleaned.csv
Order Payments: 103,886 rows saved to d:\HCL-GUVI\Cart_to_Insights\data\cleaned\olist_order_payments_cleaned.csv
Order Reviews: 99,224 rows saved to d:\HCL-GUVI\Cart_to_Insights\data\cleaned\olist_order_reviews_cleaned.csv
Products: 32,951 rows saved to d:\HCL-GUVI\Cart_to_Insights\data\cleaned\olist_products_cleaned.csv
Sellers: 3,095 rows saved to d:\HCL-GUVI\Cart_to_Insights\data\cleaned\olist_sellers_cleaned.csv
Category Translation: 71 rows saved to d:\HCL-GUVI\Cart_to_Insights\data\cleaned\product_category_name_translation_cleaned.csv

All cleaned datasets saved successfully.